# **Parte 2: API de lluvias** #14

### Responsable: Cristian Wong

**Grupo 13 · Temporada asignada: del 1 de enero al 31 de mayo de 2022.**

**Pregunta del trabajo:** ¿El Estado declara emergencia donde más llueve?

En este notebook medimos **cuánto llovió en cada departamento** durante nuestra temporada, usando la lluvia de su **capital**. Usamos dos APIs gratuitas de [Open-Meteo](https://open-meteo.com/), que no necesitan clave:

- la **API de geocodificación**, para obtener la latitud y la longitud de cada capital, y
- la **API histórica**, para pedir la lluvia diaria de la temporada.

**Resultado:** `datos/lluvias_por_departamento.csv`, con una fila por departamento, 25 en total.

## Librerías y parámetros

Definimos una sola vez las fechas de la temporada y las direcciones (URL) que vamos a usar; si algo cambia, se cambia en un solo lugar.

> **Ojo:** Corresponde a la temporada 2022 del `2022-01-01` al `2022-05-31`.

In [1]:
import io
import re
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

# Temporada asignada al Grupo 13
INICIO = "2022-01-01"
FIN = "2022-05-31"

URL_WIKIPEDIA = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"
URL_GEOCODIFICACION = "https://geocoding-api.open-meteo.com/v1/search"
URL_HISTORICA = "https://archive-api.open-meteo.com/v1/archive"

# Wikipedia rechaza los pedidos que no se identifican como navegador (error 403)
HEADERS = {"User-Agent": "Mozilla/5.0"}

print("Fecha de la consulta:", date.today())
print("Temporada:", INICIO, "al", FIN)

Fecha de la consulta: 2026-09-30
Temporada: 2022-01-01 al 2022-05-31


## Paso 1. Tabla de departamentos de Wikipedia

Wikipedia rechaza `pd.read_html(url)` directo (error 403). Por eso primero descargamos la página con `requests`, identificándonos con un `User-Agent`, y luego le pasamos el texto a `pd.read_html` con `io.StringIO`.

In [2]:
respuesta = requests.get(URL_WIKIPEDIA, headers=HEADERS, timeout=30)
print("Código de estado:", respuesta.status_code)   # 200 = todo bien

tablas = pd.read_html(io.StringIO(respuesta.text))
print("Tablas encontradas en la página:", len(tablas))
for i, tabla in enumerate(tablas):
    print(f"  Tabla {i}: {tabla.shape[0]:>2} filas | columnas: {list(tabla.columns)[:6]}")

Código de estado: 200
Tablas encontradas en la página: 6
  Tabla 0:  1 filas | columnas: ['Departamentos del Perú (con las provincias de régimen especial)']
  Tabla 1: 24 filas | columnas: ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
  Tabla 2:  8 filas | columnas: ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
  Tabla 3:  2 filas | columnas: ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
  Tabla 4:  6 filas | columnas: ['Provincia', 'Capital', 'Creación', 'Desaparición']
  Tabla 5:  1 filas | columnas: [0, 1]


`pd.read_html` devuelve **una lista** con todas las tablas de la página. La que necesitamos es la de la sección *"Departamentos del Perú"*: es la única que tiene las columnas `Departamento` y `Nombre oficial` (la de *"Antiguos departamentos"* también tiene `Departamento` y `Capital`, pero son departamentos que ya no existen).

La elegimos por sus columnas y no por su posición en la lista, para que el código no se rompa si Wikipedia agrega otra tabla antes. Nos quedamos solo con `Departamento` y `Capital`, y les ponemos nombres en minúsculas.

In [3]:
tabla_departamentos = [t for t in tablas if "Departamento" in t.columns and "Nombre oficial" in t.columns][0]

departamentos = tabla_departamentos[["Departamento", "Capital"]].copy()
departamentos.columns = ["departamento", "capital"]
departamentos

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## Paso 2. Verificación: 25 filas y capitales sin texto extra

La tabla final debe tener **25 filas**: los 24 departamentos y el Callao. Contamos las filas y comparamos los nombres con la lista oficial de departamentos del enunciado.

In [4]:
DEPARTAMENTOS_OFICIALES = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                           "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                           "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                           "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                           "San Martín", "Tacna", "Tumbes", "Ucayali"]

nombres_wikipedia = departamentos["departamento"].tolist()
print("Filas en la tabla de Wikipedia:", len(departamentos))
print("Faltan:", [d for d in DEPARTAMENTOS_OFICIALES if d not in nombres_wikipedia])
print("Sobran o están escritos distinto:", [d for d in nombres_wikipedia if d not in DEPARTAMENTOS_OFICIALES])

Filas en la tabla de Wikipedia: 24
Faltan: ['Callao']
Sobran o están escritos distinto: []


**Falta el Callao.** Tiene sentido, ya que el Callao **no es un departamento**, es la *Provincia Constitucional del Callao*, una provincia de régimen especial que no pertenece a ningún departamento. Por eso Wikipedia no la pone en la tabla de departamentos sino en otra tabla **de la misma página**, la de la sección *"Provincias de régimen especial"*. De ahí lo sacamos.

In [5]:
tabla_provincias = [t for t in tablas if "Provincia" in t.columns and "Nombre oficial" in t.columns][0]
tabla_provincias[["Provincia", "Nombre oficial", "Capital"]]

,Provincia,Nombre oficial,Capital
0,Callao[nota 6],Provincia Constitucional del Callao,Callao
1,Lima[nota 7],Provincia de Lima,Lima


Esa tabla trae dos provincias: el **Callao** y la **Provincia de Lima**. Solo agregamos el Callao. La Provincia de Lima (Lima Metropolitana) ya es parte del departamento de Lima, así que agregarla duplicaría a Lima.

In [6]:
callao = tabla_provincias[tabla_provincias["Provincia"].str.startswith("Callao")][["Provincia", "Capital"]].copy()
callao.columns = ["departamento", "capital"]

departamentos = pd.concat([departamentos, callao], ignore_index=True)
print("Filas después de agregar el Callao:", len(departamentos))
departamentos.tail(3)

Filas después de agregar el Callao: 25


,departamento,capital
22,Tumbes,Tumbes
23,Ucayali,Pucallpa
24,Callao[nota 6],Callao


Ahora revisamos la columna `capital` y de paso `departamento` . Buscamos celdas con texto extra entre paréntesis .

In [7]:
tiene_texto_extra = (departamentos["departamento"].str.contains(r"[\(\[]")
                     | departamentos["capital"].str.contains(r"[\(\[]"))
departamentos[tiene_texto_extra]

,departamento,capital
13,Lima,Huacho (de facto)
24,Callao[nota 6],Callao


Hay dos celdas con texto extra:

- `Callao[nota 6]`: es la llamada a una nota al pie de Wikipedia. Hay que quitarla para que el nombre sea igual al de la lista oficial.
- `Huacho (de facto)`: es la capital de Lima. Con ese texto la API de geocodificación **no encuentra nada**. Lo comprobamos:

In [8]:
for nombre in ["Huacho (de facto)", "Huacho"]:
    time.sleep(1)   # pausa entre pedidos
    r = requests.get(URL_GEOCODIFICACION, params={"name": nombre, "count": 10, "language": "es"}, timeout=30)
    print(f"{nombre!r:22} -> {len(r.json().get('results', []))} resultados")

'Huacho (de facto)'    -> 0 resultados
'Huacho'               -> 10 resultados


#### Decisión: para Lima usamos **Huacho**

¿Por qué Wikipedia dice *"Huacho (de facto)"*? Porque el departamento de Lima tiene dos "capitales" posibles:

- **La ciudad de Lima** es la capital del país y de la *Provincia de Lima*, que tiene régimen especial. Según la nota 7 de Wikipedia, esa provincia *"de facto, no es administrada por el Gobierno Regional de Lima; de iure, es parte del Departamento de Lima"*.
- **Huacho** es la sede del **Gobierno Regional de Lima**, que administra las otras nueve provincias del departamento (Barranca, Cajatambo, Canta, Cañete, Huaral, Huarochirí, Huaura, Oyón y Yauyos). Por eso Wikipedia la pone como capital *de facto*.

Elegimos **Huacho** porque:

1. Es la ciudad que la propia tabla de Wikipedia da como capital: solo le quitamos la aclaración *"(de facto)"*, que es justo lo que impedía que la API la encontrara.
2. La API la ubica en `admin1` = *"Departamento de Lima"* . A la ciudad de Lima, en cambio, la ubica en *"Provincia de Lima"*: la trata como una unidad aparte, igual que Wikipedia.
3. La ciudad de Lima está pegada al Callao y ambas caen en **la misma celda de la grilla** del modelo de lluvia, así que Lima y Callao tendrían exactamente los mismos datos. Lo comprobamos en el **Anexo**, al final, donde también se ve que con cualquiera de las dos ciudades la lluvia de la temporada es muy baja : la decisión no cambia el panorama.

Para limpiar las dos columnas usamos una función que borra lo que está entre corchetes o entre paréntesis:

In [9]:
def limpiar(texto):
    """Quita notas entre corchetes ('[nota 6]') y aclaraciones entre paréntesis ('(de facto)')."""
    return re.sub(r"\[.*?\]|\(.*?\)", "", texto).strip()


departamentos["departamento"] = departamentos["departamento"].apply(limpiar)
departamentos["capital"] = departamentos["capital"].apply(limpiar)

departamentos[departamentos["departamento"].isin(["Lima", "Callao"])]

,departamento,capital
13,Lima,Huacho
24,Callao,Callao


**Verificación final de la tabla.** Además la ordenamos alfabéticamente sin tomar en cuenta las tildes; así queda en el mismo orden que los ubigeos del INEI (01 Amazonas, 02 Áncash, …, 07 Callao, …, 25 Ucayali).

In [10]:
SIN_TILDES = str.maketrans("áéíóúÁÉÍÓÚ", "aeiouAEIOU")


def quitar_tildes(texto):
    return texto.translate(SIN_TILDES)


departamentos = departamentos.sort_values("departamento", key=lambda col: col.map(quitar_tildes)).reset_index(drop=True)

assert len(departamentos) == 25, "La tabla no tiene 25 filas"
assert departamentos["departamento"].is_unique, "Hay departamentos repetidos"
assert sorted(departamentos["departamento"]) == sorted(DEPARTAMENTOS_OFICIALES), "Hay nombres distintos a la lista oficial"
assert not departamentos["capital"].str.contains(r"[\(\)\[\]]").any(), "Quedan capitales con texto extra"

print("✔ 25 filas, sin repetidos, con los nombres de la lista oficial y capitales sin texto extra")
departamentos

✔ 25 filas, sin repetidos, con los nombres de la lista oficial y capitales sin texto extra


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Callao,Callao
7,Cusco,Cusco
8,Huancavelica,Huancavelica
9,Huánuco,Huánuco


## Paso 3. Latitud y longitud de cada capital (API de geocodificación)

Escribimos dos funciones pequeñas:

- `buscar_ciudad(nombre)`: le pide a la API hasta 10 lugares con ese nombre (`count=10`, `language=es`).
- `primero_en_peru(resultados)`: recorre los resultados **en el orden en que llegan** y devuelve el primero cuyo `country_code` sea `"PE"`.

Primero las probamos con Chachapoyas, el ejemplo del enunciado, y miramos todo lo que devuelve la API.

In [11]:
def buscar_ciudad(nombre):
    """Pide a la API de geocodificación hasta 10 lugares con ese nombre."""
    parametros = {"name": nombre, "count": 10, "language": "es"}
    r = requests.get(URL_GEOCODIFICACION, params=parametros, timeout=30)
    r.raise_for_status()
    return r.json().get("results", [])   # si no encuentra nada, la respuesta no trae "results"


def primero_en_peru(resultados):
    """Devuelve el primer resultado que está en el Perú (country_code == "PE"), o None si no hay ninguno."""
    for resultado in resultados:
        if resultado.get("country_code") == "PE":
            return resultado
    return None


time.sleep(1)
resultados = buscar_ciudad("Chachapoyas")
pd.DataFrame(resultados)[["name", "country_code", "admin1", "admin2", "latitude", "longitude", "feature_code"]]

,name,country_code,admin1,admin2,latitude,longitude,feature_code
0,Chachapoyas,PE,Departamento de Amazonas,Provincia de Chachapoyas,-6.23169,-77.86903,PPLA
1,Destilería Chachapoyas,AR,Provincia de Salta,Departamento Capital,-24.74735,-65.38527,PPL
2,Chachapoyas,PE,Ancash,Provincia de Santa,-8.97357,-78.49754,PPL
3,Chachapoyas,PE,Departamento de Amazonas,Provincia de Chachapoyas,-6.20181,-77.85606,AIRP


Aquí ya se ve el problema del que habla el enunciado: con el nombre "Chachapoyas" la API devuelve también un lugar en **Argentina** y otro **"Chachapoyas" en Áncash** (un poblado de la provincia del Santa), además del aeropuerto. Por eso no basta con tomar el primer resultado a ciegas. En ese sentido, nos quedamos con el primero del Perú y después **verificamos el departamento**.

La columna `feature_code` también ayuda: `PPLA` significa *capital de una división administrativa de primer nivel* (en el Perú, capital de departamento), `PPL` es un poblado cualquiera y `AIRP`, un aeropuerto.

Ahora lo hacemos para las 25 capitales, con una pausa de 1 segundo entre pedidos. Guardamos también cuántos resultados devolvió la API y cuántos eran del Perú, para ver qué tan ambiguos eran los nombres.

In [12]:
filas = []
for _, fila in departamentos.iterrows():
    time.sleep(1)   # pausa entre pedidos
    resultados = buscar_ciudad(fila["capital"])
    elegido = primero_en_peru(resultados)
    if elegido is None:
        print("⚠ No hubo resultados en el Perú para", fila["capital"])
        continue
    filas.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "admin1": elegido.get("admin1"),
        "latitud": elegido["latitude"],
        "longitud": elegido["longitude"],
        "country_code": elegido["country_code"],
        "feature_code": elegido.get("feature_code"),
        "resultados_api": len(resultados),
        "resultados_peru": sum(1 for x in resultados if x.get("country_code") == "PE"),
    })

geo = pd.DataFrame(filas)
print("Capitales geocodificadas:", len(geo))
geo.head()

Capitales geocodificadas: 25


,departamento,capital,admin1,latitud,longitud,country_code,feature_code,resultados_api,resultados_peru
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,PE,PPLA,4,3
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,PE,PPLA,1,1
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,PE,PPLA,1,1
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,PE,PPLA,9,7
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,PE,PPLA,10,1


## Paso 4. Verificación de la geocodificación

`admin1` no siempre viene escrito igual que nuestro departamento (*"Departamento de Cusco"*, *"Ancash"* sin tilde, *"Provincia Constitucional del Callao"*). Para compararlos, "normalizamos" los dos textos: minúsculas, sin tildes y sin los prefijos *"Departamento de/del"* y *"Provincia Constitucional del"*. Si después de eso son iguales, la capital está en el departamento correcto (columna `coincide`).

In [13]:
def normalizar(texto):
    """Minúsculas, sin tildes y sin prefijos como 'Departamento de', para poder comparar nombres."""
    texto = quitar_tildes(str(texto)).lower()
    for prefijo in ["provincia constitucional del ", "departamento del ", "departamento de "]:
        texto = texto.replace(prefijo, "")
    return texto.strip()


geo["coincide"] = geo["admin1"].apply(normalizar) == geo["departamento"].apply(normalizar)

geo[["departamento", "capital", "admin1", "latitud", "longitud", "coincide"]]

,departamento,capital,admin1,latitud,longitud,coincide
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,True
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,True
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,True
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,True
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,True
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,True
6,Callao,Callao,Provincia Constitucional del Callao,-12.05162,-77.13452,True
7,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,True
8,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,True
9,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,True


In [14]:
print("Filas:", len(geo))
print("¿Todas en el Perú?        ", (geo["country_code"] == "PE").all())
print("¿Todas en su departamento?", geo["coincide"].all(), f"({geo['coincide'].sum()} de {len(geo)})")
print("Tipo de lugar elegido:    ", geo["feature_code"].value_counts().to_dict())
print()
print("Búsquedas con más de un resultado:       ", (geo["resultados_api"] > 1).sum())
print("Búsquedas con resultados fuera del Perú: ", (geo["resultados_api"] > geo["resultados_peru"]).sum())
print("Búsquedas con varios lugares en el Perú: ", (geo["resultados_peru"] > 1).sum())

assert len(geo) == 25, "No se geocodificaron las 25 capitales"
assert (geo["country_code"] == "PE").all(), "Hay capitales fuera del Perú"
assert geo["coincide"].all(), "Hay capitales en un departamento equivocado"

Filas: 25
¿Todas en el Perú?         True
¿Todas en su departamento? True (25 de 25)
Tipo de lugar elegido:     {'PPLA': 25}

Búsquedas con más de un resultado:        23
Búsquedas con resultados fuera del Perú:  13
Búsquedas con varios lugares en el Perú:  21


### Cómo elegimos el resultado correcto y hacemos la revisión fila por fila

1. La API devuelve una lista de lugares con ese nombre, empezando por los que considera más relevantes. La recorrimos y nos quedamos con **el primero con `country_code == "PE"`**. Así se descartan, por ejemplo, Lima (Ohio, EE. UU.), Trujillo (Venezuela, Honduras, España), Ayacucho (Argentina) o Cajamarca (Colombia).
2. Luego revisamos que `admin1` corresponda al departamento, porque **dentro del Perú también hay homónimos**: hay "Arequipa" en Cajamarca, Loreto, San Martín y Ayacucho; "Huacho" en Áncash, Huánuco y Junín; "Chachapoyas" en Áncash; y hasta un cerro llamado "Cerro de Pasco" en Lima (San Juan de Lurigancho). En las 25 capitales, el primer resultado peruano fue el correcto.
3. Revisamos a mano las filas cuyo `admin1` no dice exactamente *"Departamento de …"*:

| departamento | `admin1` que devolvió la API | ¿Es correcto? |
|---|---|---|
| Áncash | Ancash | Sí, solo le falta la tilde |
| Apurímac | Apurímac | Sí, viene sin el prefijo "Departamento de" |
| Puno | Puno | Sí, sin el prefijo |
| Ucayali | Ucayali | Sí, sin el prefijo |
| Callao | Provincia Constitucional del Callao | Sí, es el nombre oficial del Callao |
| Lima | Departamento de Lima | Sí: es Huacho (ver la decisión del paso 2) |

   Las otras 19 filas dicen *"Departamento de X"*, con X igual a nuestro departamento.
   
4. Como control extra, las 25 ciudades elegidas tienen `feature_code = "PPLA"`: son **capitales de departamento** y no poblados homónimos (que aparecen como `PPL`). La columna `coincide` es `True` en las 25 filas: **todas están en el Perú y en el departamento correcto**.

## Paso 5. Lluvia diaria de la temporada (API histórica)

Para cada capital pedimos la variable `precipitation_sum` (lluvia total del día, en mm) del `2022-01-01` al `2022-05-31`, con `timezone=America/Lima` para que cada día vaya de medianoche a medianoche en hora peruana.

La función devuelve la respuesta completa. Primero la probamos con Chachapoyas para ver la URL que se arma y la forma del JSON.

In [15]:
def pedir_lluvia(latitud, longitud):
    """Pide a la API histórica la lluvia diaria de la temporada en un punto."""
    parametros = {
        "latitude": latitud,
        "longitude": longitud,
        "start_date": INICIO,
        "end_date": FIN,
        "daily": "precipitation_sum",
        "timezone": "America/Lima",
    }
    r = requests.get(URL_HISTORICA, params=parametros, timeout=60)
    r.raise_for_status()
    return r


time.sleep(1)
prueba = pedir_lluvia(geo.loc[0, "latitud"], geo.loc[0, "longitud"])
print("URL:", prueba.url)
datos_prueba = prueba.json()
print("Llaves del JSON:", list(datos_prueba.keys()))
print("Unidades:", datos_prueba["daily_units"])
pd.DataFrame(datos_prueba["daily"]).head()

URL: https://archive-api.open-meteo.com/v1/archive?latitude=-6.23169&longitude=-77.86903&start_date=2022-01-01&end_date=2022-05-31&daily=precipitation_sum&timezone=America%2FLima
Llaves del JSON: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']
Unidades: {'time': 'iso8601', 'precipitation_sum': 'mm'}


,time,precipitation_sum
0,2022-01-01,3.1
1,2022-01-02,3.8
2,2022-01-03,0.9
3,2022-01-04,2.5
4,2022-01-05,1.2


La respuesta trae `daily` con dos listas del mismo largo: `time` (las fechas) y `precipitation_sum` (los mm de cada día). 
También trae `latitude` y `longitude`: son las coordenadas del **centro de la celda de la grilla** del modelo que usa la API, que puede estar a algunos kilómetros de la capital.

Ahora pedimos las 25 capitales, con **una pausa de 1 segundo antes de cada pedido**, y guardamos cada respuesta en el diccionario `respuestas` (departamento → JSON).

In [16]:
respuestas = {}
for _, fila in geo.iterrows():
    time.sleep(1)   # pausa de al menos 1 segundo entre pedidos
    respuestas[fila["departamento"]] = pedir_lluvia(fila["latitud"], fila["longitud"]).json()
    dias = len(respuestas[fila["departamento"]]["daily"]["time"])
    print(f"✔ {fila['departamento']:<14} ({fila['capital']}): {dias} días")

✔ Amazonas       (Chachapoyas): 151 días
✔ Áncash         (Huaraz): 151 días
✔ Apurímac       (Abancay): 151 días
✔ Arequipa       (Arequipa): 151 días
✔ Ayacucho       (Ayacucho): 151 días
✔ Cajamarca      (Cajamarca): 151 días
✔ Callao         (Callao): 151 días
✔ Cusco          (Cusco): 151 días
✔ Huancavelica   (Huancavelica): 151 días
✔ Huánuco        (Huánuco): 151 días
✔ Ica            (Ica): 151 días
✔ Junín          (Huancayo): 151 días
✔ La Libertad    (Trujillo): 151 días
✔ Lambayeque     (Chiclayo): 151 días
✔ Lima           (Huacho): 151 días
✔ Loreto         (Iquitos): 151 días
✔ Madre de Dios  (Puerto Maldonado): 151 días
✔ Moquegua       (Moquegua): 151 días
✔ Pasco          (Cerro de Pasco): 151 días
✔ Piura          (Piura): 151 días
✔ Puno           (Puno): 151 días
✔ San Martín     (Moyobamba): 151 días
✔ Tacna          (Tacna): 151 días
✔ Tumbes         (Tumbes): 151 días
✔ Ucayali        (Pucallpa): 151 días


Juntamos todo en una sola tabla "larga": una fila por departamento y por día. Verificamos que cada departamento tenga los **151 días** de la temporada (31 + 28 + 31 + 30 + 31; 2022 no fue bisiesto).

In [17]:
tablas_diarias = []
for departamento, datos in respuestas.items():
    diaria = pd.DataFrame(datos["daily"])          # columnas: time y precipitation_sum
    diaria.insert(0, "departamento", departamento)
    tablas_diarias.append(diaria)

lluvia_diaria = pd.concat(tablas_diarias, ignore_index=True)
lluvia_diaria = lluvia_diaria.rename(columns={"time": "fecha", "precipitation_sum": "lluvia_mm"})

dias_esperados = len(pd.date_range(INICIO, FIN))
conteo = lluvia_diaria.groupby("departamento")["fecha"].agg(["count", "min", "max"])
completos = (conteo["count"] == dias_esperados) & (conteo["min"] == INICIO) & (conteo["max"] == FIN)

print("Filas de la tabla:", len(lluvia_diaria), f"(25 departamentos x {dias_esperados} días)")
print(f"¿Los 25 tienen los {dias_esperados} días, del {INICIO} al {FIN}?", completos.all())
lluvia_diaria.head()

Filas de la tabla: 3775 (25 departamentos x 151 días)
¿Los 25 tienen los 151 días, del 2022-01-01 al 2022-05-31? True


,departamento,fecha,lluvia_mm
0,Amazonas,2022-01-01,3.1
1,Amazonas,2022-01-02,3.8
2,Amazonas,2022-01-03,0.9
3,Amazonas,2022-01-04,2.5
4,Amazonas,2022-01-05,1.2


## Paso 6. Lluvia total y días de lluvia fuerte por departamento

- `lluvia_total_mm`: suma de la lluvia diaria de la temporada.
- `dias_lluvia_fuerte`: número de días con **20 mm o más** (`>= 20`).

Primero marcamos cada día como lluvia fuerte (`True`/`False`) y luego agrupamos por departamento; al sumar una columna de `True`/`False`, cada `True` cuenta como 1. La tabla se muestra ordenada de mayor a menor lluvia.

In [18]:
lluvia_diaria["lluvia_fuerte"] = lluvia_diaria["lluvia_mm"] >= 20

resumen = lluvia_diaria.groupby("departamento", as_index=False).agg(
    lluvia_total_mm=("lluvia_mm", "sum"),
    dias_lluvia_fuerte=("lluvia_fuerte", "sum"),
)
resumen["lluvia_total_mm"] = resumen["lluvia_total_mm"].round(1)

resumen.sort_values("lluvia_total_mm", ascending=False).reset_index(drop=True)

,departamento,lluvia_total_mm,dias_lluvia_fuerte
0,Loreto,1581.8,25
1,Ucayali,1288.8,20
2,Áncash,975.3,7
3,Junín,910.4,5
4,Madre de Dios,908.0,13
5,Huancavelica,803.8,1
6,San Martín,789.8,4
7,Puno,779.5,3
8,Pasco,774.0,0
9,Cajamarca,773.3,1


Comprobamos el cálculo a mano para un departamento, directamente con la lista que devolvió la API (sin pandas):

In [19]:
lista = respuestas["Amazonas"]["daily"]["precipitation_sum"]
total_a_mano = round(sum(v for v in lista if v is not None), 1)
fuertes_a_mano = sum(1 for v in lista if v is not None and v >= 20)

fila_resumen = resumen[resumen["departamento"] == "Amazonas"].iloc[0]
print("Amazonas, a mano:    ", total_a_mano, "mm |", fuertes_a_mano, "días de lluvia fuerte")
print("Amazonas, con pandas:", fila_resumen["lluvia_total_mm"], "mm |", fila_resumen["dias_lluvia_fuerte"], "días de lluvia fuerte")

Amazonas, a mano:     436.5 mm | 1 días de lluvia fuerte
Amazonas, con pandas: 436.5 mm | 1 días de lluvia fuerte


## Paso 7. Verificación: días sin dato (`None`)

Si la API no tiene el dato de un día, en la lista `precipitation_sum` llega `None` y en pandas se convierte en `NaN`. Los contamos directamente en las respuestas de la API y, para confirmar, también en la tabla de pandas.

In [20]:
sin_dato = pd.DataFrame({
    "departamento": list(respuestas.keys()),
    "dias_recibidos": [len(datos["daily"]["precipitation_sum"]) for datos in respuestas.values()],
    "dias_sin_dato": [datos["daily"]["precipitation_sum"].count(None) for datos in respuestas.values()],
})

print("Días recibidos en total:", sin_dato["dias_recibidos"].sum())
print("Días sin dato (None) en las respuestas de la API:", sin_dato["dias_sin_dato"].sum())
print("Días sin dato (NaN) en la tabla de pandas:       ", lluvia_diaria["lluvia_mm"].isna().sum())
print("Departamentos con algún día sin dato:", sin_dato.loc[sin_dato["dias_sin_dato"] > 0, "departamento"].tolist())

Días recibidos en total: 3775
Días sin dato (None) en las respuestas de la API: 0
Días sin dato (NaN) en la tabla de pandas:        0
Departamentos con algún día sin dato: []


**Resultado: 0 días sin dato.** Los 25 departamentos tienen los 151 días completos (3775 días en total), así que `lluvia_total_mm` y `dias_lluvia_fuerte` se calcularon sobre la temporada completa y no hubo que corregir nada.

Si hubiera habido días sin dato, esto es lo que habríamos hecho:

- **No** reemplazarlos por 0, porque eso sería inventar que ese día no llovió.
- Dejarlos como `NaN`. Pandas los ignora al sumar y, al comparar `>= 20`, dan `False`; es decir, el total de ese departamento quedaría **incompleto** (subestimado).
- Por eso habríamos reportado cuántos días faltan en cada departamento y, si fueran muchos.

## Paso 8. Guardar `datos/lluvias_por_departamento.csv`

Unimos las coordenadas de cada capital del paso 3, con el resumen de lluvia del paso 6, usando el nombre del departamento, y nos quedamos con las columnas pedidas. Usamos `how="left"` para no perder ningún departamento y verificamos que no quede ninguna celda vacía. Guardamos con `encoding="utf-8-sig"` para que las tildes se vean bien si se abre el archivo en Excel.

In [21]:
lluvias_por_departamento = geo[["departamento", "capital", "latitud", "longitud"]].merge(
    resumen, on="departamento", how="left"
)

assert len(lluvias_por_departamento) == 25, "No hay 25 filas"
assert lluvias_por_departamento.notna().all().all(), "Hay celdas vacías"

Path("datos").mkdir(exist_ok=True)
lluvias_por_departamento.to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")
print("✔ Archivo guardado: datos/lluvias_por_departamento.csv")

✔ Archivo guardado: datos/lluvias_por_departamento.csv


Lo leemos de vuelta para comprobar que el archivo quedó bien: 25 filas y las 6 columnas pedidas, en orden.

In [22]:
comprobacion = pd.read_csv("datos/lluvias_por_departamento.csv")

columnas_pedidas = ["departamento", "capital", "latitud", "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]
print("Filas y columnas:", comprobacion.shape)
print("¿Columnas pedidas y en orden?", list(comprobacion.columns) == columnas_pedidas)
comprobacion

Filas y columnas: (25, 6)
¿Columnas pedidas y en orden? True


,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,436.5,1
1,Áncash,Huaraz,-9.52614,-77.52869,975.3,7
2,Apurímac,Abancay,-13.63426,-72.88380,711.8,2
3,Arequipa,Arequipa,-16.39899,-71.53747,189.9,0
4,Ayacucho,Ayacucho,-13.16380,-74.22345,485.1,0
5,Cajamarca,Cajamarca,-7.16378,-78.50027,773.3,1
6,Callao,Callao,-12.05162,-77.13452,39.2,0
7,Cusco,Cusco,-13.53188,-71.96701,710.7,3
8,Huancavelica,Huancavelica,-12.78691,-74.97298,803.8,1
9,Huánuco,Huánuco,-9.92882,-76.23989,444.8,1


## Anexo. ¿Cambia algo si para Lima usamos la ciudad de Lima en vez de Huacho?

Esta comprobación **no modifica el CSV**: solo respalda la decisión del paso 2. Pedimos la lluvia de la ciudad de Lima y la comparamos con la de Huacho y la del Callao. `celda_lat` y `celda_lon` son las coordenadas de la celda del modelo que usó la API.

In [23]:
time.sleep(1)
lima = primero_en_peru(buscar_ciudad("Lima"))
print("Ciudad de Lima según la API ->", "admin1:", lima["admin1"], "| feature_code:", lima["feature_code"])

time.sleep(1)
datos_lima = pedir_lluvia(lima["latitude"], lima["longitude"]).json()

puntos = {
    "Huacho (el que usamos para Lima)": respuestas["Lima"],
    "Ciudad de Lima (alternativa)": datos_lima,
    "Callao": respuestas["Callao"],
}
filas_comparacion = []
for punto, datos in puntos.items():
    serie = pd.Series(datos["daily"]["precipitation_sum"])
    filas_comparacion.append({
        "punto": punto,
        "celda_lat": datos["latitude"],
        "celda_lon": datos["longitude"],
        "lluvia_total_mm": round(serie.sum(), 1),
        "dias_lluvia_fuerte": int((serie >= 20).sum()),
    })

misma_serie = datos_lima["daily"]["precipitation_sum"] == respuestas["Callao"]["daily"]["precipitation_sum"]
print("¿La ciudad de Lima y el Callao tienen exactamente la misma lluvia día por día?", misma_serie)
pd.DataFrame(filas_comparacion)

Ciudad de Lima según la API -> admin1: Provincia de Lima | feature_code: PPLC
¿La ciudad de Lima y el Callao tienen exactamente la misma lluvia día por día? True


,punto,celda_lat,celda_lon,lluvia_total_mm,dias_lluvia_fuerte
0,Huacho (el que usamos para Lima),-11.072056,-77.62201,13.9,0
1,Ciudad de Lima (alternativa),-12.056238,-77.06198,39.2,0
2,Callao,-12.056238,-77.06198,39.2,0


- La API ubica a la ciudad de Lima en *"Provincia de Lima"* (con `feature_code = "PPLC"`, capital del país), no en *"Departamento de Lima"*.
- La ciudad de Lima y el Callao caen en **la misma celda** del modelo y tienen la misma lluvia día por día: si usáramos Lima, dos de nuestras 25 filas tendrían datos idénticos. Con Huacho, cada una tiene su propio dato.
- Con cualquiera de las dos ciudades la lluvia de la temporada es muy baja y **no hay ningún día de lluvia fuerte**: Lima queda entre los departamentos más secos en ambos casos. Mantenemos **Huacho**.

## Resumen

| Paso | Qué hicimos | Verificación |
|---|---|---|
| 1 | Leímos la tabla de departamentos de Wikipedia con `requests` + `pd.read_html` | 24 departamentos |
| 2 | Agregamos el Callao (tabla de provincias de régimen especial), limpiamos `[nota 6]` y `(de facto)` y usamos Huacho para Lima | 25 filas, nombres iguales a la lista oficial |
| 3 | Geocodificamos las 25 capitales y nos quedamos con el primer resultado con `country_code == "PE"` | — |
| 4 | Comparamos `admin1` con el departamento (sin tildes ni prefijos) y revisamos fila por fila | 25 de 25 en el Perú y en su departamento |
| 5 | Pedimos la lluvia diaria del 1 de enero al 31 de mayo de 2022, con pausas de 1 segundo | 151 días por departamento |
| 6 | Calculamos `lluvia_total_mm` y `dias_lluvia_fuerte` (≥ 20 mm) | cálculo comprobado a mano |
| 7 | Contamos los días sin dato (`None`) | 0 días sin dato |
| 8 | Guardamos `datos/lluvias_por_departamento.csv` | 25 filas y las 6 columnas pedidas |

**Para tener en cuenta en la Parte 3:**

- La lluvia de cada departamento es la de **un solo punto: su capital**. En departamentos con costa y sierra (por ejemplo Lima, Piura, Arequipa o Áncash), la capital puede no representar lo que llovió en los distritos declarados en emergencia.
- Los datos de la API histórica de Open-Meteo vienen de **modelos de reanálisis** (una grilla de varios kilómetros que combina observaciones con un modelo del clima), no de pluviómetros instalados en cada ciudad.